In [ ]:
###### This code is used to create the stratified sample groups (by location) to prevent bias ######

In [ ]:
### Pull in Python Packages ###
import pandas as pd
import geopandas as gpd
import numpy as np

In [ ]:
#######################################
###### Get Info For Stratified Sampling ######
######################################

In [ ]:
## Get the Dams
Selected_Dams = pd.read_csv(r"F:\Insert_File_Path_Here\Final_Dam_List.csv", engine = 'python') # Update this file path

# Get the List of the dams being used
Final_Dams_List = Selected_Dams["DamID"].unique().tolist()

# Grod
GrodDams = gpd.read_file(r"F:\Insert_File_Path_Here\Study_Dams.shp", columns=['grod_id', 'type', 'lon', 'lat', 'hilarriid', 'dataset']) # Update this file path
GrodDams_Filtered = GrodDams[GrodDams['grod_id'].isin(Final_Dams_List)]

In [ ]:
## Get the basins for geographically stratifying the dams for sampling and tuning the models
HydroBasins_04 = gpd.read_file(r"F:\Insert_File_Path_Here\Basins_for_Stratified_Sampling.shp", columns=['HYB_ID_Str']) # Update this file path

In [ ]:
# Get the Dam Subset (Dir Up vs DS Comparison)
Sample_Dams = GrodDams_Filtered[:]

# Join the sampled dams  to their intersecting HydroBasin (04)
DamBasinJoin = gpd.sjoin(HydroBasins_04, Sample_Dams, how='inner', predicate='contains')
DamBasinJoin = DamBasinJoin.reset_index()

# Group the points by watershed
PointsPerBasin = DamBasinJoin.groupby(['HYB_ID_Str']).agg({'grod_id':['count']})
PointsPerBasin.columns = ["PointCount"]
PointsPerBasin = PointsPerBasin.reset_index()

## Create Groupings 
Group1 = ['7040013550','7040394690','7040394680','7040379430','7040379310','7040388740', '7040569640']
Group2 = ['7040741760','7040686450','7040047970','7040048300','7040048410','7040049270','7040707810','7040707710','7040047850']
Group3 = ['7040192560','7040392650','7040569650','7040392520']
Group4 = ['7040686540','7040612840','7040612640', '7040402430']
Group5 = ['7040402430', '7040038340','7040039620','7040039630','7040039870']
Group6 = ['7040041050','7040041400','7040041410','7040042040','7040043090','7040043160','7040043170','7040045890','7040045900','7040046590']

BasinGroups = PointsPerBasin[:]
BasinGroups["Strat_Group"] = np.nan
BasinGroups["Strat_Group"] = np.where(BasinGroups['HYB_ID_Str'].isin(Group1), "G1", BasinGroups["Strat_Group"])
BasinGroups["Strat_Group"] = np.where(BasinGroups['HYB_ID_Str'].isin(Group2), "G2", BasinGroups["Strat_Group"])
BasinGroups["Strat_Group"] = np.where(BasinGroups['HYB_ID_Str'].isin(Group3), "G3", BasinGroups["Strat_Group"])
BasinGroups["Strat_Group"] = np.where(BasinGroups['HYB_ID_Str'].isin(Group4), "G4", BasinGroups["Strat_Group"])
BasinGroups["Strat_Group"] = np.where(BasinGroups['HYB_ID_Str'].isin(Group5), "G5", BasinGroups["Strat_Group"])
BasinGroups["Strat_Group"] = np.where(BasinGroups['HYB_ID_Str'].isin(Group6), "G6", BasinGroups["Strat_Group"])

## Group by Groupings
PointsPerGroup = BasinGroups.groupby(['Strat_Group']).agg({'PointCount':['sum']})
PointsPerGroup.columns = ["PointCount"]
PointsPerGroup = PointsPerGroup.reset_index()
PointsPerGroup['Percent'] = PointsPerGroup['PointCount']/sum(PointsPerGroup['PointCount'])
PointsPerGroup

In [ ]:
# Export the data
PointsPerGroup.to_csv(r'F:\Insert_File_Output_Path_Here\Dam_Basin_Stratified_Sampling_Percentages.csv') # Update this file path

In [ ]:
## Assign the Groups to the Dams ## 
DamBasinJoin["Strat_Group"] = np.nan
DamBasinJoin["Strat_Group"] = np.where(DamBasinJoin['HYB_ID_Str'].isin(Group1), "G1", DamBasinJoin["Strat_Group"])
DamBasinJoin["Strat_Group"] = np.where(DamBasinJoin['HYB_ID_Str'].isin(Group2), "G2", DamBasinJoin["Strat_Group"])
DamBasinJoin["Strat_Group"] = np.where(DamBasinJoin['HYB_ID_Str'].isin(Group3), "G3", DamBasinJoin["Strat_Group"])
DamBasinJoin["Strat_Group"] = np.where(DamBasinJoin['HYB_ID_Str'].isin(Group4), "G4", DamBasinJoin["Strat_Group"])
DamBasinJoin["Strat_Group"] = np.where(DamBasinJoin['HYB_ID_Str'].isin(Group5), "G5", DamBasinJoin["Strat_Group"])
DamBasinJoin["Strat_Group"] = np.where(DamBasinJoin['HYB_ID_Str'].isin(Group6), "G6", DamBasinJoin["Strat_Group"])

# Clean up the file
Dam_Basin_Strat_Sampling = DamBasinJoin[["grod_id", "HYB_ID_Str","Strat_Group"]]
Dam_Basin_Strat_Sampling= Dam_Basin_Strat_Sampling.sort_values(by='grod_id', ascending=True)
Dam_Basin_Strat_Sampling = Dam_Basin_Strat_Sampling.reset_index(drop = True)
Dam_Basin_Strat_Sampling =Dam_Basin_Strat_Sampling.rename(columns={"grod_id":"DamID", "HYB_ID_Str":"BasinID"})
Dam_Basin_Strat_Sampling['DamID'] = Dam_Basin_Strat_Sampling['DamID'] .astype(int)
Dam_Basin_Strat_Sampling

In [ ]:
# Export the data
Dam_Basin_Strat_Sampling.to_csv(r'F:\Insert_File_Output_Path_Here\Dam_Basin_Stratified_Sampling.csv') # Update this file path